<a href="https://colab.research.google.com/github/bahmedx/730/blob/main/Image_Classification_with_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Image Classification with Convolutional Neural Networks using CIFAR-10**
**Objective**

The objective of this project is to develop and evaluate Convolutional Neural Network (CNN) models for image classification using the CIFAR-10 dataset. The project investigates the impact of data augmentation, batch normalization, dropout regularizat

**Import Libraries**

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms

from torch.utils.data import DataLoader
from torch.utils.data import random_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

**Reproducibility**

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

**Data Loading**

In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2023, 0.1994, 0.2010)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD)
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD)
])

full_train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=train_transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=test_transform
)

train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_train_dataset,
    [train_size, val_size]
)

BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

classes = (
    'plane',
    'car',
    'bird',
    'cat',
    'deer',
    'dog',
    'frog',
    'horse',
    'ship',
    'truck'
)

print("Training Samples:", len(train_dataset))
print("Validation Samples:", len(val_dataset))
print("Testing Samples:", len(test_dataset))

**CNN Baseline Architecture**

In [ ]:
class ConvNet(nn.Module):

    def __init__(self, num_classes=10, dropout_rate=0.3):
        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(3,32,3,padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(32,32,3,padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(64,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*4*4,256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256,10)
        )

    def forward(self,x):
        x = self.features(x)
        x = self.classifier(x)
        return x

**Alternative Architecture**

In [ ]:
class DeepConvNet(nn.Module):

    def __init__(self, num_classes=10, dropout_rate=0.5):
        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(3,32,3,padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(32,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(64,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Conv2d(128,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(128,256,3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),

            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256*4*4,512),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(512,10)
        )

    def forward(self,x):
        x = self.features(x)
        x = self.classifier(x)
        return x

**Validation Function**

In [ ]:
def evaluate(model, loader, criterion):

    model.eval()

    loss_total = 0
    correct = 0
    total = 0

    preds = []
    labels_all = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss_total += loss.item() * images.size(0)

            _, predicted = outputs.max(1)

            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

            preds.extend(predicted.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())

    acc = 100 * correct / total
    avg_loss = loss_total / total

    return avg_loss, acc, np.array(preds), np.array(labels_all)

**Training Function**

In [ ]:
def train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    scheduler,
    epochs=15
):

    history = {
        'train_loss':[],
        'train_acc':[],
        'val_loss':[],
        'val_acc':[]
    }

    for epoch in range(epochs):

        model.train()

        running_loss = 0
        correct = 0
        total = 0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)

            _, pred = outputs.max(1)

            correct += pred.eq(labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / total
        train_acc = 100 * correct / total

        val_loss, val_acc, _, _ = evaluate(
            model,
            val_loader,
            criterion
        )

        scheduler.step()

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Train Acc: {train_acc:.2f}% | "
            f"Val Acc: {val_acc:.2f}%"
        )

    return history

**Train Baseline Model**

In [ ]:
model = ConvNet().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=15
)

history = train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    scheduler,
    epochs=15
)

**Final Test Evaluation**

In [ ]:
test_loss, test_acc, preds, labels = evaluate(
    model,
    test_loader,
    criterion
)

precision = precision_score(
    labels,
    preds,
    average='macro'
)

recall = recall_score(
    labels,
    preds,
    average='macro'
)

f1 = f1_score(
    labels,
    preds,
    average='macro'
)

print("\nFINAL TEST RESULTS")
print(f"Accuracy : {test_acc:.2f}%")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

print(
    classification_report(
        labels,
        preds,
        target_names=classes
    )
)

**Alternative Architecture Experiment**

In [ ]:
deep_model = DeepConvNet().to(device)

optimizer2 = optim.Adam(
    deep_model.parameters(),
    lr=0.001
)

scheduler2 = optim.lr_scheduler.CosineAnnealingLR(
    optimizer2,
    T_max=10
)

deep_history = train_model(
    deep_model,
    train_loader,
    val_loader,
    criterion,
    optimizer2,
    scheduler2,
    epochs=10
)

_, deep_acc, _, _ = evaluate(
    deep_model,
    test_loader,
    criterion
)

print("Deep CNN Test Accuracy:", deep_acc)

**Model Comparison**

In [ ]:
results = pd.DataFrame({
    "Model":[
        "Baseline CNN",
        "Deep CNN"
    ],
    "Test Accuracy":[
        test_acc,
        deep_acc
    ]
})

results.sort_values(
    "Test Accuracy",
    ascending=False
)